# **0 <font color='orange'>|</font> Install & Import**
---

In [ ]:
# Install
!pip install -q -U langchain-community langchain-openai langchain-chroma
!pip install -q tiktoken pypdf
!pip install -q unstructured[all-docs]
!pip install -q -U gradio

In [ ]:
#
# Hilfslösung für Fehler mit OpenAI 1.56.0 - Client.__init__() hat ein unerwartetes Schlüsselwortargument 'proxyes' erhalten
#
# Ende mit Fehler: ERROR: pip's dependency resolver does not currently take into account -- weitere Ausführung klappt trotzdem
#
!pip install openai==1.55.3 httpx==0.27.2 --force-reinstall --quiet
import os

os.kill(os.getpid(), 9)

In [ ]:
# Standardbibliotheken
import os
import re

# Drittanbieterbibliotheken
import gradio as gr
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain.document_loaders import (
    DirectoryLoader,
    PyPDFLoader,
    UnstructuredWordDocumentLoader,
)
from langchain_community.document_loaders import UnstructuredMarkdownLoader
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_core.prompts import ChatPromptTemplate
from langchain_chroma import Chroma

In [ ]:
# Warnung ausstellen
import warnings

warnings.filterwarnings("ignore")

# **1 <font color='orange'>|</font> Übergreifende Parameter**
---

In [ ]:
# OpenAI API-Schlüssel setzen
from google.colab import userdata

os.environ["OPENAI_API_KEY"] = userdata.get("OpenAI-API-Key")

In [ ]:
# Verzeichnis mit den Dateien
directory_path = "/content/Dateien"
persist_directory = "/content/Vektordatenbank"

In [ ]:
# Chat-Verlauf initialisieren
chat_history = []

# **2 <font color='orange'>|</font> Import Dateien**
---

In [ ]:
# Kopieren der Dateien von Github
!mkdir Dateien
!curl -L https://raw.githubusercontent.com/ralf-42/ML_Intro/main/02%20data/_rag/_files/Der-Weihnachtsabend.pdf -o Dateien/Der-Weihnachtsabend.pdf
!curl -L https://raw.githubusercontent.com/ralf-42/ML_Intro/main/02%20data/_rag/_files/In-den-tiefen-Waeldern.docx -o Dateien/In-den-tiefen-Waeldern.docx
!curl -L https://raw.githubusercontent.com/ralf-42/ML_Intro/main/02%20data/_rag/_files/Git-Grundbegriffe.md -o Dateien/Git-Grundbegriffe.md

In [ ]:
# Mapping von Dateitypen zu Loaders
LOADER_MAPPING = {
    "*.md": UnstructuredMarkdownLoader,
    "*.docx": UnstructuredWordDocumentLoader,
    "*.pdf": PyPDFLoader,
}

In [ ]:
# Funktion lädt Dokumente aus dem  Verzeichnis basierend auf den unterstützten Dateitypen
def load_documents_from_directory(directory_path):
    documents = []

    # Iteriert über alle unterstützten Dateitypen und deren Loader
    for file_pattern, loader_cls in LOADER_MAPPING.items():
        loader = DirectoryLoader(
            directory_path,
            glob=file_pattern,
            loader_cls=loader_cls,
        )
        documents.extend(loader.load())  # Geladene Dokumente hinzufügen

    return documents

In [ ]:
# Dokumente laden
documents = load_documents_from_directory(directory_path)

In [ ]:
# Festlegen des Text-Splitters: große Textmengen werden in handhabbare Abschnitte unterteilt, die kontextuelle Integrität des Inhalts bleibt bewahrt
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
docs = text_splitter.split_documents(documents)

# **3 <font color='orange'>|</font> Einbettungen & VektorDB**
---

In [ ]:
# Embeddingsmodell festlegen
embeddings = OpenAIEmbeddings()

In [ ]:
# Vektordatenbank erstellen und speichern
vectorstore = Chroma.from_documents(
    docs, embeddings, persist_directory=persist_directory
)

In [ ]:
# Vektordatenbank laden
# embeddings = OpenAIEmbeddings()
# vectorstore = Chroma(persist_directory=persist_directory, embedding_function=embeddings)
# print("Vektordatenbank erfolgreich geladen.")

# **4 <font color='orange'>|</font> RetrievalQA-Kette**
---

In [ ]:
# Festlegen LLM und Retriever
llm = ChatOpenAI(temperature=0, model="gpt-3.5-turbo")
retriever = vectorstore.as_retriever()

In [ ]:
# Template/Vorlage der Eingaben
prompt = ChatPromptTemplate.from_template(
    """Beantworten Sie die folgende Frage ausschließlich auf der Grundlage des bereitgestellten Kontexts, antworte in deutscher Sprache:


<context>
{context}
</context>

Question: {input}"""
)

In [ ]:
# Kette basierend auf einer gegebenen Aufforderung (Prompt) neue Dokumente zu generieren
document_chain = create_stuff_documents_chain(llm, prompt)

In [ ]:
# Diese Kette ist dafür zuständig, relevante Informationen aus einer Wissensbasis abzurufen und diese Informationen zu nutzen, um eine Anfrage zu beantworten.
retrieval_chain = create_retrieval_chain(retriever, document_chain)

# **5 <font color='orange'>|</font> App-Integration ChatBot**
---

In [ ]:
# Chat mit OpenAI & Vektordatenbank
def chat_openai(message, chat_history=[]):

    # Frage an die Retrieval-Kette senden
    result = retrieval_chain.invoke({"input": message})
    answer = result["answer"]
    source_docs = result.get("source_documents", [])

    # Quellen extrahieren (Dateinamen) wenn gefunden
    content_not_found = r"\bnot provide\b|\bnot found\b|\bnot contain\b|\bkeine Informationen\b|\bno answer\b|\bnot a valid question\b"
    not_found = re.findall(content_not_found, answer)
    if not_found != []:
        sources_text = "keine Quellen"
    else:
        sources = [
            os.path.basename(doc.metadata["source"]) for doc in result["context"]
        ]
        sources = set(sources)
        sources_text = "\n".join(f"- {source}" for source in sources)

    # Antwort mit Quellenangaben formatieren
    full_response = f"{answer}\n\n**Quellen:**\n{sources_text}"

    return full_response

In [ ]:
# chat_openai("Wer ist Merlin?")

# **6 <font color='orange'>|</font> App Design**
---

In [ ]:
demo = gr.ChatInterface(
    fn=chat_openai,
    type="messages",
    title="📚 Dokumentenbasierter Chatbot mit Quellenangabe",
    description="\n\n*Der Chatbot wertet die verfügbaren Dokumente aus und beantwortet Fragen zum Inhalt*",
)

# **7 <font color='orange'>|</font> Launch**
---

In [ ]:
demo.launch()